<a href="https://colab.research.google.com/github/younguk072023/Thermal-Imaging/blob/main/real_yolo5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Google Drive에 있는 파일 목록 확인
!ls /content/drive/MyDrive


In [ ]:
!git clone https://github.com/ultralytics/yolov5.git
%cd yolov5
!pip install -r requirements.txt


In [ ]:
import zipfile
import os

# 압축 파일 경로 설정 (Google Drive에 있는 압축 파일 경로)
zip_file_path = '/content/drive/MyDrive/data.zip'

# 압축 해제 경로 설정
unzip_dir = '/content/dataset'

# 압축 파일 해제
with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
    zip_ref.extractall(unzip_dir)

# 압축 해제된 파일 구조 확인
print("압축 해제된 파일 목록:")
print(os.listdir(unzip_dir))


In [ ]:
data_yaml = """
train: /content/dataset/data/images/train
val: /content/dataset/data/images/val

nc: 17  # 클래스 수 (0부터 16까지)
names: [
    'Class0',
    'Class1',
    'Class2',
    'Class3',
    'Class4',
    'Class5',
    'Class6',
    'Class7',
    'Class8',
    'Class9',
    'Class10',
    'Class11',
    'Class12',
    'Class13',
    'Class14',
    'CriticalHeat',  # 15번 클래스
    'ModerateHeat'   # 16번 클래스
]
"""

# data.yaml 파일 저장
with open('data.yaml', 'w') as f:
    f.write(data_yaml)


In [ ]:
!python train.py --img 256 --batch 16 --epochs 100 --data data.yaml --weights yolov5x.pt

In [ ]:
!python detect.py --weights runs/train/exp/weights/best.pt --img 256 --conf 0.01 --source /content/dataset/data/images/val


In [ ]:
from IPython.display import Image, display

# 원본 이미지 시각화
display(Image(filename='/content/dataset/data/images/val/a0_aug_0.jpg'))


In [ ]:
import os

# 라벨 파일 경로 확인
label_train_dir = '/content/dataset/data/labels/train'
label_val_dir = '/content/dataset/data/labels/val'

# 경로 내 파일 목록 출력
print("Train 라벨 파일 목록:", os.listdir(label_train_dir))
print("Val 라벨 파일 목록:", os.listdir(label_val_dir))


In [ ]:
import torch
import cv2
import matplotlib.pyplot as plt
from google.colab.patches import cv2_imshow  # Colab에서 cv2.imshow() 대신 사용

# 모델 로드 (학습된 best.pt 파일을 사용)
model = torch.hub.load('ultralytics/yolov5', 'custom', path='runs/train/exp/weights/best.pt')

# 테스트할 이미지 경로 설정
img_path = '/content/dataset/data/images/val/a0_aug_0.jpg'  # 테스트할 이미지 파일의 경로로 바꿔주세요.

# 이미지 로드
img = cv2.imread(img_path)

# 클래스별 색상 설정
class_colors = {
    'ModerateHeat': (0, 255, 0),  # 초록색
    'CriticalHeat': (0, 0, 255)   # 빨간색
}

# 이미지가 잘 로드되었는지 확인
if img is None:
    print("이미지를 로드할 수 없습니다. 경로를 확인해주세요.")
else:
    # 모델을 사용하여 예측
    results = model(img)

    # 감지된 객체 수 확인
    detected_objects = results.xyxy[0]  # 감지된 객체의 바운딩 박스 정보 가져오기
    if len(detected_objects) == 0:
        print("모델이 객체를 감지하지 못했습니다.")
    else:
        print(f"{len(detected_objects)}개의 객체가 감지되었습니다.")

        # 바운딩 박스와 함께 정확도 출력
        for *box, conf, cls in detected_objects:  # 각 객체에 대해
            class_name = model.names[int(cls)]
            label = f'{class_name} {conf:.2f}'  # 클래스 이름과 정확도
            color = class_colors.get(class_name, (255, 255, 255))  # 해당 클래스의 색상, 기본값은 흰색
            print(f"클래스: {class_name}, 정확도: {conf:.2f}, 바운딩 박스 좌표: {box}")

            # 바운딩 박스와 라벨을 이미지에 그리기
            x1, y1, x2, y2 = map(int, box)
            cv2.rectangle(img, (x1, y1), (x2, y2), color, 2)
            cv2.putText(img, label, (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 1, cv2.LINE_AA)  # 폰트 크기를 0.5로 조정

        # 결과 이미지 저장 및 출력
        output_path = '/content/detected_image_with_boxes.jpg'
        cv2.imwrite(output_path, img)

        # Matplotlib을 사용하여 이미지 출력 (BGR 이미지를 RGB로 변환)
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        plt.figure(figsize=(12, 8))
        plt.imshow(img_rgb)
        plt.axis('off')
        plt.show()

        print(f"검출된 이미지를 저장했습니다: {output_path}")

        # Colab에서 이미지 출력
        detected_img = cv2.imread(output_path)
        cv2_imshow(detected_img)  # cv2.imshow 대신 cv2_imshow 사용


In [ ]:
import matplotlib.pyplot as plt

# 모델 이름 (순서: YOLOv5s, YOLOv5m, YOLOv5l, YOLOv5x)
models = ['YOLOv5s', 'YOLOv5m', 'YOLOv5l', 'YOLOv5x']

# 각 모델에 대한 성능 데이터 (S, M, L, X 순서)
precision = [0.785, 0.773, 0.806, 0.771]
recall = [0.797, 0.792, 0.811, 0.758]
map50 = [0.774, 0.761, 0.803, 0.756]
map5095 = [0.375, 0.365, 0.37, 0.364]

# 그래프 그리기
plt.figure(figsize=(10, 6))

plt.plot(models, precision, marker='o', label='Precision', color='blue')
plt.plot(models, recall, marker='v', label='Recall', color='green')
plt.plot(models, map50, marker='s', label='mAP50', color='orange')
plt.plot(models, map5095, marker='d', label='mAP50-95', color='red')

# 그래프의 축과 제목 설정
plt.xlabel('Model', fontsize=12)
plt.ylabel('Performance Metrics', fontsize=12)
plt.title('YOLOv5 Model Performance Comparison', fontsize=14)

# 축 범위 설정
plt.ylim(0.3, 0.9)

# 격자 추가
plt.grid(True)

# 범례 추가
plt.legend()

# 그래프 출력
plt.tight_layout()
plt.show()


In [ ]:
import os
import torch
import cv2
import matplotlib.pyplot as plt
from google.colab.patches import cv2_imshow  # Colab에서 cv2.imshow() 대신 사용

# 테스트 이미지와 라벨 경로 설정
test_images_dir = '/content/dataset/data/images/test'
test_labels_dir = '/content/dataset/data/labels/test'

# 테스트할 이미지 파일 리스트 가져오기
test_image_files = os.listdir(test_images_dir)

# 모델 로드 (학습된 best.pt 파일을 사용)
model = torch.hub.load('ultralytics/yolov5', 'custom', path='runs/train/exp/weights/best.pt')

# 클래스별 색상 설정
class_colors = {
    'CriticalHeat': (0, 0, 255),  # 빨간색
    'ModerateHeat': (0, 255, 0)   # 초록색
}

# 테스트 데이터셋의 이미지로 성능 평가
for img_file in test_image_files:
    img_path = os.path.join(test_images_dir, img_file)

    # 이미지 로드
    img = cv2.imread(img_path)

    # 이미지가 잘 로드되었는지 확인
    if img is None:
        print(f"이미지를 로드할 수 없습니다: {img_path}")
        continue

    # 모델을 사용하여 예측
    results = model(img)

    # 감지된 객체 수 확인
    detected_objects = results.xyxy[0]  # 감지된 객체의 바운딩 박스 정보 가져오기
    if len(detected_objects) == 0:
        print(f"모델이 객체를 감지하지 못했습니다: {img_file}")
    else:
        print(f"{img_file}: {len(detected_objects)}개의 객체가 감지되었습니다.")

        # 바운딩 박스와 함께 정확도 출력
        for *box, conf, cls in detected_objects:  # 각 객체에 대해
            class_name = model.names[int(cls)]
            label = f'{class_name}'# {conf:.2f}' # 클래스 이름과 정확도 표시
            color = class_colors.get(class_name, (255, 255, 255))  # 해당 클래스의 색상, 기본값은 흰색
            print(f"클래스: {class_name}, 정확도: {conf:.2f}, 바운딩 박스 좌표: {box}")
            print(f"클래스: {class_name}, 정확도: {conf:.2f}, 바운딩 박스 좌표: {box}")

            # 바운딩 박스와 라벨을 이미지에 그리기
            x1, y1, x2, y2 = map(int, box)
            cv2.rectangle(img, (x1, y1), (x2, y2), color, 2)  # 바운딩 박스 그리기
            cv2.putText(img, label, (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 1, cv2.LINE_AA)

        # 결과 이미지 저장 및 출력
        output_path = f'/content/test_results/{img_file}'
        os.makedirs('/content/test_results', exist_ok=True)
        cv2.imwrite(output_path, img)

        # Colab에서 이미지 출력
        detected_img = cv2.imread(output_path)
        cv2_imshow(detected_img)

print("테스트 데이터셋에 대한 성능 평가가 완료되었습니다.")


In [ ]:
import os
import torch
import cv2
import numpy as np
from google.colab.patches import cv2_imshow  # Colab에서 cv2.imshow() 대신 사용

# 테스트 이미지와 라벨 경로 설정
test_images_dir = '/content/dataset/data/images/test'
test_labels_dir = '/content/dataset/data/labels/test'

# 테스트할 이미지 파일 리스트 가져오기
test_image_files = os.listdir(test_images_dir)

# 모델 로드 (학습된 best.pt 파일을 사용)
model = torch.hub.load('ultralytics/yolov5', 'custom', path='runs/train/exp/weights/best.pt')

# 클래스별 색상 설정
class_colors = {
    'CriticalHeat': (0, 0, 255),  # 빨간색
    'ModerateHeat': (0, 255, 0)   # 초록색
}

# 성능 기록을 위한 변수 초기화
total_detected = 0
total_confidence_sum = 0.0
total_objects = 0

# 테스트 데이터셋의 이미지로 성능 평가
for img_file in test_image_files:
    img_path = os.path.join(test_images_dir, img_file)

    # 이미지 로드
    img = cv2.imread(img_path)

    # 이미지가 잘 로드되었는지 확인
    if img is None:
        print(f"이미지를 로드할 수 없습니다: {img_path}")
        continue

    # 모델을 사용하여 예측
    results = model(img)

    # 감지된 객체 수 확인
    detected_objects = results.xyxy[0]  # 감지된 객체의 바운딩 박스 정보 가져오기
    detected_count = len(detected_objects)
    if detected_count == 0:
        print(f"모델이 객체를 감지하지 못했습니다: {img_file}")
    else:
        print(f"{img_file}: {detected_count}개의 객체가 감지되었습니다.")
        total_detected += detected_count  # 감지된 객체 수 누적

        # 각 객체의 정보 기록
        for *box, conf, cls in detected_objects:
            class_name = model.names[int(cls)]
            total_confidence_sum += conf  # 감지된 객체의 정확도 합산
            total_objects += 1  # 총 객체 수 증가
            label = f'{class_name}'  # 클래스 이름과 정확도 표시
            color = class_colors.get(class_name, (255, 255, 255))  # 해당 클래스의 색상, 기본값은 흰색
            x1, y1, x2, y2 = map(int, box)
            cv2.rectangle(img, (x1, y1), (x2, y2), color, 2)  # 바운딩 박스 그리기
            cv2.putText(img, label, (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 1, cv2.LINE_AA)

        # 결과 이미지 저장 및 출력
        output_path = f'/content/test_results/{img_file}'
        os.makedirs('/content/test_results', exist_ok=True)
        cv2.imwrite(output_path, img)

        # Colab에서 이미지 출력
        detected_img = cv2.imread(output_path)
        cv2_imshow(detected_img)

# 전체 평균 성능 계산
average_confidence = total_confidence_sum / total_objects if total_objects > 0 else 0
print("\n테스트 데이터셋에 대한 성능 평가가 완료되었습니다.")
print(f"총 감지된 객체 수: {total_detected}")
print(f"평균 정확도: {average_confidence:.2f}")


In [ ]:
import os
import torch
import cv2
import pandas as pd
from google.colab.patches import cv2_imshow

# 테스트 이미지와 라벨 경로 설정
test_images_dir = '/content/dataset/data/images/test'
test_image_files = os.listdir(test_images_dir)

# 모델 로드 (학습된 best.pt 파일 사용)
model = torch.hub.load('ultralytics/yolov5', 'custom', path='runs/train/exp/weights/best.pt')

# 클래스별 성능 결과 저장할 변수 초기화
class_performance = {
    'CriticalHeat': {'instances': 0, 'precision': [], 'recall': [], 'mAP50': []},
    'ModerateHeat': {'instances': 0, 'precision': [], 'recall': [], 'mAP50': []}
}

# 테스트 데이터셋의 이미지로 성능 평가
for img_file in test_image_files:
    img_path = os.path.join(test_images_dir, img_file)

    # 이미지 로드
    img = cv2.imread(img_path)

    if img is None:
        print(f"이미지를 로드할 수 없습니다: {img_path}")
        continue

    # 모델을 사용하여 예측
    results = model(img)

    detected_objects = results.xyxy[0]
    if len(detected_objects) == 0:
        print(f"모델이 객체를 감지하지 못했습니다: {img_file}")
        continue

    for *box, conf, cls in detected_objects:
        class_name = model.names[int(cls)]
        class_performance[class_name]['instances'] += 1
        class_performance[class_name]['precision'].append(conf.item())
        class_performance[class_name]['recall'].append(conf.item())
        class_performance[class_name]['mAP50'].append(conf.item())

        # 바운딩 박스 그리기
        x1, y1, x2, y2 = map(int, box)
        color = (0, 0, 255) if class_name == 'CriticalHeat' else (0, 255, 0)
        label = class_name  # 수치값 제거
        cv2.rectangle(img, (x1, y1), (x2, y2), color, 2)
        cv2.putText(img, label, (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 1, cv2.LINE_AA)

        output_path = f'/content/test_results/{img_file}'
        os.makedirs('/content/test_results', exist_ok=True)
        cv2.imwrite(output_path, img)
        cv2_imshow(cv2.imread(output_path))

# 성능 결과 요약
summary = {
    'Class': [],
    'Images': [],
    'Instances': [],
    'P': [],
    'R': [],
    'mAP50': [],
}

for class_name, metrics in class_performance.items():
    summary['Class'].append(class_name)
    summary['Images'].append(len(test_image_files))
    summary['Instances'].append(metrics['instances'])
    summary['P'].append(sum(metrics['precision']) / len(metrics['precision']) if metrics['precision'] else 0)
    summary['R'].append(sum(metrics['recall']) / len(metrics['recall']) if metrics['recall'] else 0)
    summary['mAP50'].append(sum(metrics['mAP50']) / len(metrics['mAP50']) if metrics['mAP50'] else 0)

# 평가표를 데이터프레임으로 변환하여 출력
df = pd.DataFrame(summary)
print("\n성능 평가표:")
print(df)

# CSV 파일로 저장
df.to_csv('/content/test_results/performance_summary.csv', index=False)
print("\n성능 평가표가 'performance_summary.csv' 파일로 저장되었습니다.")


In [ ]:
!ls runs/train/yolov5m_experiment/weights/
